# 02 — Exploratory Data Analysis on Cleaned Data

This notebook should be run **after `01_cleaning.ipynb`**. It loads `data/processed/cleaned.csv` and performs EDA on the cleaned dataset used for statistical testing and modeling.

Initial raw-data quality checks, impossible-value flagging, and cleaning summaries are handled in `01_cleaning.ipynb`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.viz.plots import plot_target_balance
from src.config import TARGET, FIGURES_DIR, TABLES_DIR, PROCESSED_DIR

sns.set_theme(style='whitegrid')

In [ ]:
cleaned_path = PROCESSED_DIR / 'cleaned.csv'

if not cleaned_path.exists():
    raise FileNotFoundError(
        f'{cleaned_path} was not found. Run 01_cleaning.ipynb first to create cleaned.csv.'
    )

df = pd.read_csv(cleaned_path)
df.shape

## Confirm cleaned dataset

The dataset below should no longer include `patient_id`, and clinically impossible values should already be converted to missing values by the cleaning notebook.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# split columns into numeric, binary and categorical
num = df.select_dtypes('number').drop(columns=[TARGET]).columns.tolist()
binary = [c for c in num if df[c].dropna().isin([0, 1]).all()]
num = [c for c in num if c not in binary]
cat = df.select_dtypes(exclude='number').columns.tolist() + binary
print('numeric:', num)
print('categorical:', cat)

In [ ]:
types = pd.DataFrame({'variable': df.columns})
types['kind'] = types['variable'].apply(
    lambda c: 'target' if c == TARGET else 'numeric' if c in num else 'categorical')
types['dtype'] = types['variable'].map(df.dtypes.astype(str))
types['missing'] = types['variable'].map(df.isna().sum())
types.to_csv(TABLES_DIR / 'tbl_variable_types.csv', index=False)
types

In [ ]:
summary = df[num].describe().T.round(2)
summary.to_csv(TABLES_DIR / 'tbl_numeric_summary.csv')
summary

In [ ]:
rows = []
for c in cat:
    for level, count in df[c].value_counts().items():
        rows.append({'variable': c, 'level': level, 'count': int(count),
                     'pct': round(count / len(df) * 100, 2)})
freq = pd.DataFrame(rows)
freq.to_csv(TABLES_DIR / 'tbl_categorical_freq.csv', index=False)
freq

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
plot_target_balance(df, TARGET, ax=ax)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_target_balance.png', dpi=120)
df[TARGET].value_counts(normalize=True).round(3)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, c in zip(axes.ravel(), num):
    sns.histplot(df[c].dropna(), kde=True, ax=ax)
    ax.set_title(c)
for ax in axes.ravel()[len(num):]:
    ax.axis('off')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_hist_numeric.png', dpi=120)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, c in zip(axes.ravel(), num):
    sns.boxplot(y=df[c], ax=ax)
    ax.set_title(c)
for ax in axes.ravel()[len(num):]:
    ax.axis('off')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_box_numeric.png', dpi=120)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, c in zip(axes.ravel(), num):
    sns.boxplot(x=df[TARGET], y=df[c], ax=ax)
    ax.set_title(f'{c} by {TARGET}')
for ax in axes.ravel()[len(num):]:
    ax.axis('off')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_box_by_target.png', dpi=120)

In [ ]:
# violin plots show the full distribution shape per readmission group
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, c in zip(axes.ravel(), num):
    sns.violinplot(x=df[TARGET], y=df[c], inner='quartile', ax=ax)
    ax.set_title(f'{c} by {TARGET}')
for ax in axes.ravel()[len(num):]:
    ax.axis('off')
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, len(cat), figsize=(4 * len(cat), 4))
for ax, c in zip(np.ravel(axes), cat):
    df.groupby(c)[TARGET].mean().plot.bar(ax=ax, color='indianred')
    ax.set_ylabel('readmit rate')
    ax.set_title(c)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_bar_categorical.png', dpi=120)

In [ ]:
# proportion of each readmission class within every categorical level
for c in cat:
    prop = pd.crosstab(df[c], df[TARGET], normalize='index')
    prop.plot(kind='bar', stacked=True, figsize=(6, 3), colormap='coolwarm')
    plt.title(f'{c} vs {TARGET} (proportions)')
    plt.ylabel('proportion')
    plt.xticks(rotation=0)
    plt.legend(title=TARGET, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
corr = df[num + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation matrix')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_corr_heatmap.png', dpi=120)
corr.round(2).to_csv(TABLES_DIR / 'tbl_correlations.csv')

In [ ]:
c = df[num].corr().abs()
np.fill_diagonal(c.values, 0)
a, b = c.stack().idxmax()
fig, ax = plt.subplots(figsize=(6, 5))
sns.regplot(x=df[a], y=df[b], ax=ax,
            scatter_kws={'alpha': 0.3, 's': 12}, line_kws={'color': 'red'})
ax.set_title(f'{a} vs {b} (r = {df[a].corr(df[b]):.2f})')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_scatter_reg.png', dpi=120)

## Outliers and encoding preview

Outliers are reviewed using the IQR rule, but unusual values are not automatically removed here. For this healthcare dataset, extreme values may be clinically meaningful unless they violate predefined impossible-value rules from the cleaning notebook.

In [ ]:
# IQR rule: count values more than 1.5*IQR outside the quartiles
# These are reviewed as potential outliers, not automatically removed.
rows = []
for c in num:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows.append({
        'feature': c,
        'lower': round(lo, 2),
        'upper': round(hi, 2),
        'n_outliers': int(((df[c] < lo) | (df[c] > hi)).sum())
    })

outlier_summary = pd.DataFrame(rows)
outlier_summary.to_csv(TABLES_DIR / 'tbl_outlier_summary_cleaned.csv', index=False)
outlier_summary

In [ ]:
# Preview what one-hot encoding does to categorical text columns.
# Final encoding should happen inside the modeling pipeline to avoid data leakage.
obj_cols = df.select_dtypes(exclude='number').columns.tolist()
encoded = pd.get_dummies(df, columns=obj_cols, drop_first=True)
print('shape before:', df.shape, '-> after one-hot:', encoded.shape)
encoded.head()